Data downloading and batching test file

In [1]:
import numpy as np
import h5py
import xarray as xr
import torch
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from pathlib import Path

from aurora.batch import Batch, Metadata

/Users/apple/miniconda3/envs/aurora/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Data will be downloaded here.
download_path = Path("./data/downloads")

In [3]:
import cdsapi

c = cdsapi.Client()

download_path = download_path.expanduser()
download_path.mkdir(parents=True, exist_ok=True)

In [4]:
# Download the static variables.
if not (download_path / "static.nc").exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "geopotential",
                "land_sea_mask",
                "soil_type",
            ],
            "year": "2020",
            "month": "01",
            "day": "01",
            "time": "00:00",
            "format": "netcdf",
        },
        str(download_path / "static.nc"),
    )
print("Static variables downloaded!")

# Download the surface-level variables.
if not (download_path / "2020-01-01-surface-level.nc").exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "2m_temperature",
                "10m_u_component_of_wind",
                "10m_v_component_of_wind",
                "mean_sea_level_pressure",
            ],
            "year": "2020",
            "month": "01",
            "day": "01",
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "format": "netcdf",
        },
        str(download_path / "2020-01-01-surface-level.nc"),
    )
print("Surface-level variables downloaded!")

# Download the atmospheric variables.
if not (download_path / "2020-01-01-atmospheric.nc").exists():
    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "temperature",
                "u_component_of_wind",
                "v_component_of_wind",
                "specific_humidity",
                "geopotential",
            ],
            "pressure_level": [
                "50",
                "100",
                "150",
                "200",
                "250",
                "300",
                "400",
                "500",
                "600",
                "700",
                "850",
                "925",
                "1000",
            ],
            "year": "2020",
            "month": "01",
            "day": "01",
            "time": ["00:00", "06:00", "12:00", "18:00"],
            "format": "netcdf",
        },
        str(download_path / "2020-01-01-atmospheric.nc"),
    )
print("Atmospheric variables downloaded!")


2025-11-23 18:32:06,739 INFO Request ID is 68f4c2cd-93ec-44f9-89cd-dc720fe0ce39
2025-11-23 18:32:06,910 INFO status has been updated to accepted
2025-11-23 18:32:13,546 INFO status has been updated to running
2025-11-23 18:32:18,075 INFO status has been updated to successful


Static variables downloaded!


2025-11-23 18:32:21,454 INFO Request ID is bd922703-c74f-4981-83dc-f4e16d31adcd
2025-11-23 18:32:21,614 INFO status has been updated to accepted
2025-11-23 18:32:35,661 INFO status has been updated to running
2025-11-23 18:32:43,427 INFO status has been updated to successful


Surface-level variables downloaded!


2025-11-23 18:32:49,644 INFO Request ID is 76f52a5e-b46e-4bc4-98e9-b99d2b7c8009
2025-11-23 18:32:49,825 INFO status has been updated to accepted
2025-11-23 18:33:04,031 INFO status has been updated to successful
                                                                                        

Atmospheric variables downloaded!


In [5]:
static_vars_ds = xr.open_dataset(download_path / "static.nc", engine="netcdf4")
static_vars_ds = static_vars_ds.sel(latitude=static_vars_ds.latitude[:720])
surf_vars_ds = xr.open_dataset(download_path / "2020-01-01-surface-level.nc", engine="netcdf4")
surf_vars_ds = surf_vars_ds.sel(latitude=surf_vars_ds.latitude[:720])
atmos_vars_ds = xr.open_dataset(download_path / "2020-01-01-atmospheric.nc", engine="netcdf4")
atmos_vars_ds = atmos_vars_ds.sel(latitude=atmos_vars_ds.latitude[:720])

batch = Batch(
    surf_vars={
        # First select the first two time points: 00:00 and 06:00. Afterwards, `[None]`
        # inserts a batch dimension of size one.
        "2t": torch.from_numpy(surf_vars_ds["t2m"].values[:2][None]),
        "10u": torch.from_numpy(surf_vars_ds["u10"].values[:2][None]),
        "10v": torch.from_numpy(surf_vars_ds["v10"].values[:2][None]),
        "msl": torch.from_numpy(surf_vars_ds["msl"].values[:2][None]),
    },
    static_vars={
        # The static variables are constant, so we just get them for the first time.
        "z": torch.from_numpy(static_vars_ds["z"].values[0]),
        "slt": torch.from_numpy(static_vars_ds["slt"].values[0]),
        "lsm": torch.from_numpy(static_vars_ds["lsm"].values[0]),
    },
    atmos_vars={
        "t": torch.from_numpy(atmos_vars_ds["t"].values[:2][None]),
        "u": torch.from_numpy(atmos_vars_ds["u"].values[:2][None]),
        "v": torch.from_numpy(atmos_vars_ds["v"].values[:2][None]),
        "q": torch.from_numpy(atmos_vars_ds["q"].values[:2][None]),
        "z": torch.from_numpy(atmos_vars_ds["z"].values[:2][None]),
    },
    metadata=Metadata(
        lat=torch.from_numpy(surf_vars_ds.latitude.values),
        lon=torch.from_numpy(surf_vars_ds.longitude.values),
        # Converting to `datetime64[s]` ensures that the output of `tolist()` gives
        # `datetime.datetime`s. Note that this needs to be a tuple of length one:
        # one value for every batch element. Select element 1, corresponding to time
        # 06:00.
        time=(surf_vars_ds.valid_time.values.astype("datetime64[s]").tolist()[1],),
        atmos_levels=tuple(int(level) for level in atmos_vars_ds.pressure_level.values),
    ),
)

Following is for visualization

In [ ]:
# Hydrological variables appear to be only used for visualization, 
# so zipping doesn't matter that much
# Download the surface-level hydrological variables.
if not (download_path / "2020-01-01-hydrological.nc").exists():
    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "potential_evaporation",
                "runoff",
                "volumetric_soil_water_layer_1",
                "volumetric_soil_water_layer_2",
                "volumetric_soil_water_layer_3",
            ],
            "year": "2020",
            "month": "01",
            "day": "01",
            "time": ["00:00", "01:00", "02:00", "03:00", "04:00", "05:00", 
                     "06:00", "07:00", "08:00", "09:00", "10:00", "11:00",
                     "12:00", "13:00", "14:00", "15:00", "16:00", "17:00", 
                     "18:00", "19:00", "20:00", "21:00", "22:00", "23:00"],
            "format": "netcdf",
        },
        str(download_path / "2020-01-01-hydrological.nc"),
    )
print("Hydrological variables downloaded!")

import zipfile

file_path = download_path / "2020-01-01-hydrological.nc"

# If file is ZIP, extract the .nc inside it
if zipfile.is_zipfile(file_path):
    with zipfile.ZipFile(file_path, 'r') as z:
        z.extractall(download_path)
    print("Extracted netCDF files from ZIP.")

Extracted netCDF files from ZIP.
